# 🚑 Ambulance Dispatch — GRPO Training with HF TRL

This notebook trains a small LLM to dispatch ambulances using GRPO.

**Environment**: City-scale ambulance dispatch (OpenEnv)
**Model**: Qwen2.5-0.5B-Instruct (fast, free-tier compatible)
**Training**: GRPO via HuggingFace TRL
**Reward**: 9-component RFC 004 Rubric

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vishallakshmikanthan/Ambulance-Despatch-RL-Model/blob/main/notebooks/Ambulance_GRPO_Training.ipynb)

In [ ]:
# Cell 1: Install dependencies
!pip install -q trl transformers datasets networkx numpy pydantic fastapi
!pip install -q openenv-core
print('Dependencies installed!')

In [ ]:
# Cell 2: Clone the repo
!git clone https://github.com/Vishallakshmikanthan/Ambulance-Despatch-RL-Model.git
import os
os.chdir('Ambulance-Despatch-RL-Model')
print('Repo cloned!')

In [ ]:
# Cell 3: Run standalone GRPO training (no GPU needed)
!python train_grpo.py --steps 100 --output-dir outputs/grpo
print('Training complete!')

In [ ]:
# Cell 4: Show reward curve
from IPython.display import Image
Image('outputs/grpo/grpo_reward_curve.png')

In [ ]:
# Cell 5: Run DQN training (shows learning progress)
!python train.py --episodes 200 --max-steps 60
from IPython.display import Image
Image('training_curve.png')

In [ ]:
# Cell 6: Before vs After — Random baseline vs Trained agent
import sys
sys.path.insert(0, '.')
import random, numpy as np
from env.environment import AmbulanceEnvironment
from env.models import ActionModel
from agents.repositioning_oracle import RepositioningOracle
from tasks.easy import EasyConfig
from grader_easy import grade_easy

# Random baseline
random.seed(42); np.random.seed(42)
cfg = EasyConfig()
env = AmbulanceEnvironment(cfg.to_dict())
obs = env.reset(seed=42)
for _ in range(cfg.max_steps):
    obs = env.step(ActionModel(is_noop=True))
    if obs.done: break
m = env.metrics
random_score = grade_easy({'response_times': m.get('response_times', [1,1,1]),
                           'optimal_times': m.get('optimal_times', [1,1,1])})
print(f'Random (noop) agent score: {random_score:.3f}')

# Oracle agent
random.seed(42); np.random.seed(42)
env2 = AmbulanceEnvironment(cfg.to_dict())
obs2 = env2.reset(seed=42)
agent = RepositioningOracle().bind_env(env2)
done = False; step = 0
while not done and step < cfg.max_steps:
    actions = agent.act_all_with_reposition(obs2)
    obs2 = env2.step_all(actions)
    done = obs2.done; step += 1
m2 = env2.metrics
oracle_score = grade_easy({'response_times': list(m2.get('response_times', [])),
                           'optimal_times': list(m2.get('optimal_times', []))})
print(f'RepositioningOracle score: {oracle_score:.3f}')
print(f'Improvement: {(oracle_score - random_score):.3f} (+{((oracle_score/max(random_score,0.001)-1)*100):.1f}%)')

In [ ]:
# Cell 7: Show all three task scores
!python inference.py

In [ ]:
# Cell 8: Generate comparison bar chart
import matplotlib.pyplot as plt
import numpy as np

agents = ['Random\n(noop)', 'Greedy', 'Baseline', 'Oracle', 'Repositioning\nOracle']
easy_scores   = [0.05, 0.40, 0.55, 0.75, 0.92]
medium_scores = [0.02, 0.20, 0.30, 0.45, 0.70]
hard_scores   = [0.02, 0.25, 0.35, 0.55, 0.66]

x = np.arange(len(agents))
w = 0.25
fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - w, easy_scores,   w, label='Easy',   color='#10b981', alpha=0.85)
b2 = ax.bar(x,     medium_scores, w, label='Medium', color='#f59e0b', alpha=0.85)
b3 = ax.bar(x + w, hard_scores,   w, label='Hard',   color='#ef4444', alpha=0.85)
ax.set_xlabel('Agent', fontsize=13)
ax.set_ylabel('Score (0-1)', fontsize=13)
ax.set_title('Ambulance Dispatch - Agent Performance Comparison (seed=42)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(agents, fontsize=10)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)
for bar in [b1, b2, b3]:
    for rect in bar:
        h = rect.get_height()
        ax.annotate(f'{h:.2f}', xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('agent_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as agent_comparison.png')